In [0]:
from pyspark.sql import functions as F 

In [0]:
df = spark.read.table('silver.loans')

In [0]:
%sql
use schema gold

In [0]:
df.write.mode("overwrite").saveAsTable("pre_gold")

In [0]:
%sql
-- What is the portfolio breakdown by loan status (Active, Defaulted, Overdue, Closed)?

create or replace table gold.loan_distribution 
as 

with loan_distribution as 
(select loan_status, 
       round((sum(loan_amount) / 10000000),2) as portfolio_value_in_Cr,
       count(*) as No_of_loans
from gold.pre_gold
group by loan_status
order by sum(loan_amount) desc)

select loan_status, portfolio_value_in_Cr, No_of_loans, 
       round((No_Of_loans * 100.0 / sum(No_of_loans) over()),2) as pct_contribution
from loan_distribution

In [0]:
%sql
select * 
from gold.loan_distribution

In [0]:
%sql
-- Which customers have multiple loans and what's their repayment behavior?

create or replace table gold.customers_with_multiple_loans
as

with loan_distribution as 
(select customer_id , 
        loan_status, 
        count(*) as No_of_loans_in_that_status, 
        round((sum(loan_amount) / 100000),2) as Total_amount_in_that_status_in_lakhs
from gold.pre_gold
group by customer_id , loan_status)

, total_loan_addon as
(select *, sum(No_of_loans_in_that_status) over(partition by customer_id) as Total_loans 
from loan_distribution)

select customer_id,
       loan_status,
       No_of_loans_in_that_status,
       Total_amount_in_that_status_in_lakhs,
       Total_loans
from total_loan_addon
where Total_loans > 1


In [0]:
%sql
select * 
from gold.customers_with_multiple_loans

In [0]:
%sql
-- What is the portfolio performance by loan purpose (Course Fees, Living Expenses, etc.)?
create or replace table gold.loan_purpose
as 

with loan_purpose as 
(select purpose_of_loan, round((sum(loan_amount) /10000000),2) as Loan_value_in_Cr_per_purpose
from gold.pre_gold
group by purpose_of_loan)

select *, round((Loan_value_in_Cr_per_purpose * 100.0 / sum(Loan_value_in_Cr_per_purpose) over()),2) as pct_contribution
from loan_purpose

In [0]:
%sql
select * 
from gold.loan_purpose

In [0]:
%sql
-- What are the executive KPIs (total portfolio value, default rate, average interest rate)?
create or replace table gold.executive_KPIs 
as

select round((sum(loan_amount) / 10000000),2) as portfolio_value_in_Cr, 
       round(sum(case when loan_status = 'Defaulted' then loan_amount / 10000000 end),2) as total_default_amount_in_Cr,
       round(sum(case when loan_status = 'Defaulted' then 1 else 0 end),2) as default_loans_count,
       count(distinct loan_id) as total_loans,
       round(sum(case when loan_status = 'Defaulted' then 1 else 0 end) * 100.0 / count(distinct loan_id),2) as default_rate,
       round(avg(interest_rate),2) as avg_interest_rate
from gold.pre_gold



In [0]:
%sql
select * 
from executive_KPIs

In [0]:
%sql
-- How does loan performance vary by loan size (small, medium, large loans)?

create or replace table loan_performance_by_size_gold 
as 

with loan_buckets as 
(select loan_id, round((loan_amount/100000),2) as loan_amount_in_lakhs, loan_status,
      case when loan_amount > 600000 then 'large'
           when loan_amount > 300000 then 'medium'
           else 'small'
      end as loan_size
from gold.pre_gold)

select loan_size, loan_status, count(*) as No_of_loans, round(sum(loan_amount_in_lakhs)/100,2) as Total_loan_amount_in_Cr
from loan_buckets
group by loan_size, loan_status
order by loan_size, loan_status

In [0]:
%sql
select * 
from loan_performance_by_size_gold